<a href="https://colab.research.google.com/github/IsraelVessel/AI-Assignment/blob/main/week7AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Load the COMPAS dataset
try:
    df = pd.read_csv('compas-scores-two-years.csv')
    print("Dataset loaded successfully.")



# Identify sensitive attributes and the outcome variable
sensitive_attributes = ['sex', 'race', 'age_cat'] # Common sensitive attributes in COMPAS
outcome_variable = 'two_year_recid' # Common outcome variable in COMPAS

print(f"\nSensitive attributes identified: {sensitive_attributes}")
print(f"Outcome variable identified: {outcome_variable}")

# Review the first few rows and data types if the DataFrame loaded successfully
if df is not None:
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())
    print("\nData types of the DataFrame columns:")
    print(df.info())

Error: 'compas-scores-two-years.csv' not found. Please ensure the file is in the correct directory.

Sensitive attributes identified: ['sex', 'race', 'age_cat']
Outcome variable identified: two_year_recid


In [2]:
import requests

url = 'https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv'
file_name = 'compas-scores-two-years.csv'

try:
    response = requests.get(url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    with open(file_name, 'wb') as f:
        f.write(response.content)
    print(f"'{file_name}' downloaded successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")


'compas-scores-two-years.csv' downloaded successfully.


In [3]:
import pandas as pd

# Load the COMPAS dataset
try:
    df = pd.read_csv('compas-scores-two-years.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'compas-scores-two-years.csv' not found. Please ensure the file is in the correct directory.")
    df = None # Set df to None to indicate failure to load

# Identify sensitive attributes and the outcome variable
sensitive_attributes = ['sex', 'race', 'age_cat'] # Common sensitive attributes in COMPAS
outcome_variable = 'two_year_recid' # Common outcome variable in COMPAS

print(f"\nSensitive attributes identified: {sensitive_attributes}")
print(f"Outcome variable identified: {outcome_variable}")

# Review the first few rows and data types if the DataFrame loaded successfully
if df is not None:
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())
    print("\nData types of the DataFrame columns:")
    print(df.info())

Dataset loaded successfully.

Sensitive attributes identified: ['sex', 'race', 'age_cat']
Outcome variable identified: two_year_recid

First 5 rows of the DataFrame:
   id                name   first         last compas_screening_date   sex  \
0   1    miguel hernandez  miguel    hernandez            2013-08-14  Male   
1   3         kevon dixon   kevon        dixon            2013-01-27  Male   
2   4            ed philo      ed        philo            2013-04-14  Male   
3   5         marcu brown   marcu        brown            2013-01-13  Male   
4   6  bouthy pierrelouis  bouthy  pierrelouis            2013-03-26  Male   

          dob  age          age_cat              race  ...  v_decile_score  \
0  1947-04-18   69  Greater than 45             Other  ...               1   
1  1982-01-22   34          25 - 45  African-American  ...               1   
2  1991-05-14   24     Less than 25  African-American  ...               3   
3  1993-01-21   23     Less than 25  African-American

In [4]:
print(f"Initial number of records: {len(df)}")

# Filter out cases where c_charge_degree is 'O' (other) or 'F' (felony), keep only 'M' (misdemeanor)
df_filtered = df[df['c_charge_degree'] != 'O'].copy()
df_filtered = df_filtered[df_filtered['c_charge_degree'] != 'F'].copy()
print(f"Records after filtering 'O' and 'F' c_charge_degree: {len(df_filtered)}")

# Filter out records where 'days_b_screening_arrest' is less than -30
df_filtered = df_filtered[df_filtered['days_b_screening_arrest'] <= 30].copy()
print(f"Records after filtering days_b_screening_arrest: {len(df_filtered)}")

# Filter out records where 'is_recid' is not -1
df_filtered = df_filtered[df_filtered['is_recid'] != -1].copy()
print(f"Records after filtering is_recid: {len(df_filtered)}")

# Filter out records where 'c_jail_out' is null
df_filtered = df_filtered[df_filtered['c_jail_out'].notna()].copy()
print(f"Records after filtering c_jail_out nulls: {len(df_filtered)}")

# Filter out records where 'v_score_text' is 'N/A'
df_filtered = df_filtered[df_filtered['v_score_text'] != 'N/A'].copy()
print(f"Records after filtering 'N/A' v_score_text: {len(df_filtered)}")

# ProPublica's analysis also filters based on a specific `score_text` ('High', 'Medium', 'Low')
# and `two_year_recid` being 0 or 1. Let's ensure these are correct.
# The 'score_text' column represents the COMPAS risk score, which is typically 'Low', 'Medium', 'High'.
# Filter for relevant score_text values
df_filtered = df_filtered[df_filtered['score_text'].isin(['Low', 'Medium', 'High'])].copy()
print(f"Records after filtering score_text: {len(df_filtered)}")

# Ensure the outcome variable 'two_year_recid' only contains 0 or 1
df_filtered = df_filtered[df_filtered['two_year_recid'].isin([0, 1])].copy()
print(f"Records after ensuring two_year_recid values: {len(df_filtered)}")

# Drop irrelevant columns as per ProPublica's analysis setup
# These columns often contain redundant information, identifiers, or are not used in the final model.
columns_to_drop = [
    'id', 'name', 'first', 'last', 'compas_screening_date', 'dob', 'age',
    'c_jail_in', 'c_jail_out', 'c_case_number', 'c_offense_date',
    'c_arrest_date', 'c_days_from_compas', 'c_charge_desc',
    'r_case_number', 'r_charge_degree', 'r_days_from_arrest',
    'r_offense_date', 'r_charge_desc', 'r_jail_in', 'r_jail_out',
    'violent_recid', 'is_violent_recid', 'vr_case_number',
    'vr_charge_degree', 'vr_offense_date', 'vr_charge_desc',
    'type_of_assessment', 'score_text', 'screening_date',
    'v_type_of_assessment', 'v_score_text', 'v_screening_date',
    'in_custody', 'out_custody', 'priors_count.1', 'start', 'end', 'event'
]

# Ensure all columns to drop actually exist in the DataFrame
columns_to_drop_exist = [col for col in columns_to_drop if col in df_filtered.columns]
df_filtered = df_filtered.drop(columns=columns_to_drop_exist, errors='ignore')

print(f"Records after dropping irrelevant columns: {len(df_filtered)}")
print("Remaining columns:")
print(df_filtered.columns)

# Display the first few rows of the preprocessed DataFrame
print("\nFirst 5 rows of the preprocessed DataFrame:")
print(df_filtered.head())

# Display data types of the preprocessed DataFrame
print("\nData types of the preprocessed DataFrame columns:")
print(df_filtered.info())

Initial number of records: 7214
Records after filtering 'O' and 'F' c_charge_degree: 2548
Records after filtering days_b_screening_arrest: 2314
Records after filtering is_recid: 2314
Records after filtering c_jail_out nulls: 2314
Records after filtering 'N/A' v_score_text: 2314
Records after filtering score_text: 2314
Records after ensuring two_year_recid values: 2314
Records after dropping irrelevant columns: 2314
Remaining columns:
Index(['sex', 'age_cat', 'race', 'juv_fel_count', 'decile_score',
       'juv_misd_count', 'juv_other_count', 'priors_count',
       'days_b_screening_arrest', 'c_charge_degree', 'is_recid',
       'decile_score.1', 'v_decile_score', 'two_year_recid'],
      dtype='object')

First 5 rows of the preprocessed DataFrame:
       sex       age_cat              race  juv_fel_count  decile_score  \
5     Male       25 - 45             Other              0             1   
8   Female       25 - 45         Caucasian              0             1   
11    Male  Less 

## Conduct Fairness Audit with AI Fairness 360

### Subtask:
Implement a fairness audit on the COMPAS dataset using the AI Fairness 360 library, calculating disparate impact ratio and equal opportunity difference.


**Reasoning**:
To conduct the fairness audit as requested, I need to install the AI Fairness 360 library first, as it's not a standard library in most environments. This is a prerequisite for importing its components and performing the fairness calculations.



In [ ]:
import sys
!{sys.executable} -m pip install 'aif360[all]' --quiet

print("AI Fairness 360 library installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 35.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.8/515.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 